In [2]:
# =========================================================
# Notebook : Gold Layer Spark-native pour Gradient Boosting
# avec rééquilibrage cible (SMOTE-like)
# =========================================================

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml import Pipeline
from pyspark.sql import functions as F

# ------------------------
# Config MinIO
# ------------------------
MINIO_ACCESS_KEY = "minio"
MINIO_SECRET_KEY = "minio123"
MINIO_BUCKET = "telco-churn"
MINIO_ENDPOINT = "minio1:9000"

# ------------------------
# Spark Session
# ------------------------
print("🚀 Initialisation SparkSession pour Gold Layer...")
spark = (
    SparkSession.builder
    .appName("TelcoChurn_Gold_Spark")
    .master("spark://spark-master:7077")
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.endpoint", f"http://{MINIO_ENDPOINT}")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.delta.logStore.class", "org.apache.spark.sql.delta.storage.S3SingleDriverLogStore")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.driver.memory", "2g")
    .config("spark.executor.memory", "2g")
    .config("spark.executor.cores", "2")
    .config("spark.cores.max", "4")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"✅ Spark initialisé - App: {spark.sparkContext.applicationId}")

try:
    # ------------------------
    # Charger Silver Layer
    # ------------------------
    print("\n📂 Chargement du Silver Layer...")
    silver_path = f"s3a://{MINIO_BUCKET}/silver/telco_churn"
    df = spark.read.format("delta").load(silver_path)
    print(f"✅ Silver Layer chargé : {df.count()} lignes")

    # ------------------------
    # Création features dérivées
    # ------------------------
    print("\n🔧 Création des features dérivées...")
    df = df.withColumn("avg_charge", when(col("tenure") > 0, col("TotalCharges")/col("tenure")).otherwise(0.0))
    
    df = df.withColumn(
        "tenure_bin",
        when(col("tenure") <= 12, "0-12")
        .when((col("tenure") > 12) & (col("tenure") <= 24), "12-24")
        .when((col("tenure") > 24) & (col("tenure") <= 48), "24-48")
        .when((col("tenure") > 48) & (col("tenure") <= 60), "48-60")
        .otherwise("60+")
    )
    
    df = df.withColumn("Internet_Security", col("HasInternetService") * col("OnlineSecurity"))
    df = df.withColumn("Internet_Backup", col("HasInternetService") * col("OnlineBackup"))
    print("   ✅ Features dérivées créées")

    # ------------------------
    # Encodage catégoriel
    # ------------------------
    print("\n🔧 Encodage catégoriel...")
    
    # Colonnes déjà indexées dans Silver Layer
    existing_indexed_cols = ["Contract_idx", "PaymentMethod_idx"]
    new_categorical_cols = ["InternetService", "tenure_bin"]
    
    # Indexers uniquement pour les nouvelles colonnes
    indexers = [
        StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
        for c in new_categorical_cols
    ]
    
    # Colonnes à encoder : toutes les colonnes indexées (existantes + nouvelles)
    categorical_cols = ["Contract", "PaymentMethod"] + new_categorical_cols
    encoders = [OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_ohe") for c in categorical_cols]

    # ------------------------
    # Colonnes numériques
    # ------------------------
    num_cols = ["tenure", "MonthlyCharges", "TotalCharges", "avg_charge"]
    for c in num_cols:
        df = df.withColumn(c, col(c).cast("double"))

    assembler_num = VectorAssembler(inputCols=num_cols, outputCol="num_unscaled")
    scaler = StandardScaler(inputCol="num_unscaled", outputCol="num_scaled", withMean=False, withStd=True)

    pipeline = Pipeline(stages=indexers + encoders + [assembler_num, scaler])
    model = pipeline.fit(df)
    df_gold_tmp = model.transform(df)
    print("   ✅ Encodage et normalisation appliqués")

    # ------------------------
    # Assemblage final des features
    # ------------------------
    all_features = [f"{c}_ohe" for c in categorical_cols] + ["num_scaled"] + \
                   ["HasInternetService","HasPhoneService","MultipleLines",
                    "Partner","Dependents","PaperlessBilling",
                    "Internet_Security","Internet_Backup"]

    final_assembler = VectorAssembler(inputCols=all_features, outputCol="features")
    df_gold = final_assembler.transform(df_gold_tmp)
    print(f"   ✅ Vecteur de features assemblé : {len(all_features)} features")

    # ------------------------
    # Rééquilibrage Spark-native (Random over-sampling minoritaires)
    # ------------------------
    print("\n🔧 Rééquilibrage de la cible (SMOTE-like)...")
    counts = df_gold.groupBy("Churn").count().collect()
    count_dict = {row['Churn']: row['count'] for row in counts}
    majority = max(count_dict, key=count_dict.get)
    minority = min(count_dict, key=count_dict.get)
    ratio = count_dict[majority] // count_dict[minority] - 1

    df_minority = df_gold.filter(col("Churn") == minority)
    oversampled = df_minority
    for _ in range(ratio):
        oversampled = oversampled.union(df_minority)
    df_gold_balanced = df_gold.union(oversampled)

    balanced_counts = df_gold_balanced.groupBy("Churn").count().collect()
    print("   ✅ Rééquilibrage terminé")
    for row in balanced_counts:
        print(f"      → Classe {row['Churn']} : {row['count']} lignes")

    # ------------------------
    # Sauvegarde Gold Layer
    # ------------------------
    print("\n💾 Sauvegarde du Gold Layer...")
    gold_path = f"s3a://{MINIO_BUCKET}/gold/telco_churn"
    df_gold_balanced.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(gold_path)
    print(f"✅ SUCCÈS : Gold Layer sauvegardé")
    print(f"   → Chemin : {gold_path}")
    print(f"   → Format : Delta Lake")
    print(f"   → Lignes : {df_gold_balanced.count()}")

    # ------------------------
    # RÉSUMÉ FINAL
    # ------------------------
    print("\n" + "="*70)
    print("✅ PIPELINE GOLD LAYER SPARK-NATIVE TERMINÉ AVEC SUCCÈS !")
    print("="*70)
    print(f"📊 Données prêtes pour Gradient Boosting : {df_gold_balanced.count()} lignes")
    print(f"📁 Emplacement : {gold_path}")
    print(f"🎯 Vecteur de features : {len(all_features)} dimensions")
    print("\n💡 Prochaines étapes :")
    print("   1. Entraîner Gradient Boosting Classifier (pyspark.ml.classification.GBTClassifier)")
    print("   2. Évaluer AUC, précision, recall...")
    print("   3. Déployer le modèle en production")
    print("="*70)

except Exception as e:
    print(f"\n❌ ERREUR dans le pipeline Gold Spark-native : {e}")
    import traceback
    traceback.print_exc()
    raise

finally:
    print("\n🔴 Arrêt de la session Spark...")
    try:
        spark.stop()
        print("✅ Session arrêtée - Ressources libérées")
    except Exception as e:
        print(f"⚠️ Erreur lors de l'arrêt : {e}")


🚀 Initialisation SparkSession pour Gold Layer...
✅ Spark initialisé - App: app-20251217205456-0006

📂 Chargement du Silver Layer...


✅ Silver Layer chargé : 7041 lignes

🔧 Création des features dérivées...
   ✅ Features dérivées créées

🔧 Encodage catégoriel...


   ✅ Encodage et normalisation appliqués
   ✅ Vecteur de features assemblé : 13 features

🔧 Rééquilibrage de la cible (SMOTE-like)...


   ✅ Rééquilibrage terminé
      → Classe 0 : 5173 lignes
      → Classe 1 : 5604 lignes

💾 Sauvegarde du Gold Layer...


✅ SUCCÈS : Gold Layer sauvegardé
   → Chemin : s3a://telco-churn/gold/telco_churn
   → Format : Delta Lake
   → Lignes : 10777

✅ PIPELINE GOLD LAYER SPARK-NATIVE TERMINÉ AVEC SUCCÈS !
📊 Données prêtes pour Gradient Boosting : 10777 lignes
📁 Emplacement : s3a://telco-churn/gold/telco_churn
🎯 Vecteur de features : 13 dimensions

💡 Prochaines étapes :
   1. Entraîner Gradient Boosting Classifier (pyspark.ml.classification.GBTClassifier)
   2. Évaluer AUC, précision, recall...
   3. Déployer le modèle en production

🔴 Arrêt de la session Spark...
✅ Session arrêtée - Ressources libérées
